- 정상(N) 사기(P)
- Credit Card Fraud Detection
    - 99.89% 정상거래 0.17% 사기거래
- 중요한 평가지표 : Recall
    - 사기거래를 정상거래로 잘못 분류( FN )  - 고객피해 및 장기적으로는 비지니스의문제
    - 정상거래를 사기거래로 잘못 분류 ( FP  )  - 고객달래기.. 
- 사기거래를 놓치면 안되는 경우 - 재현율    

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,roc_curve,roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
data = fetch_openml(name='creditcard',as_frame=True,parser='auto').frame

In [ ]:
import pandas as pd
y = pd.to_numeric(data.iloc[:,-1]).values
x = data.drop('Class', axis=1).to_numpy()
x.shape, y.shape

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,stratify=y, test_size=0.2,random_state=42)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train_t = torch.FloatTensor(x_train)
y_train_t = torch.FloatTensor(y_train)
x_test_t = torch.FloatTensor(x_test)
y_test_t = torch.FloatTensor(y_test)

In [ ]:
class DetetFraud(nn.Module):
  def __init__(self, input_dim):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(input_dim,64),
        nn.ReLU(),
        nn.Linear(64,32),
        nn.ReLU(),
        nn.Linear(32,1),        
    )
  def forward(self ,x):
    return self.network(x)    

# 불균형 비율 만큼 positive(1) 클래스에 가중치 부여  - 사기
num_pos = sum(y_train == 1)
num_neg = sum(y_train == 0)
pos_weight_val = torch.tensor([num_neg / num_pos])   # 정상데이터수 / 사기데이터수

model = DetetFraud(x_train_t.shape[-1])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_val)
optimizer  = Adam(model.parameters(), lr = 0.001)

In [ ]:
from sklearn.metrics import classification_report
import torch

def evaluate_model(model, x, y):
  with model.no_grad():
    logits = model(x)
    probs = torch.sigmoid(logits).numpy()
    preds = (probs >=0.5).astype(int)
  print(f'confusion matrix : {confusion_matrix(y, preds)}')
  print(classification_report(y,preds, target_name = ['0 정상', '1 사기']))
  return probs    

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
epochs = 20
batch_size = 2048
dataset = TensorDataset(x_train_t, y_train_t)
dataloader = DataLoader(dataset,batch_size=batch_size, shuffle=True)

from tqdm import tqdm
for epoch in tqdm(epochs):
  total_loss = 0
  for batch_x, batch_y in dataloader:
    optimizer.zero_grad()
    output = model(batch_x)
    loss = criterion(output, batch_y)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
avg_loss = total_loss / len(dataloader)    
if (epoch+1) % 5 == 0:
  print(f'epoch : {epoch+1}/{epochs} 평균 loss = {avg_loss:.4f}')
